# Playground Series S6E3 - Telco Churn: A Seed-Ensemble of XGBoost Models

**Problem.** Predict whether a telecom customer will *churn* (cancel their subscription) from their contract, billing, and service-usage attributes. This is a binary classification task scored with **ROC AUC** on the [Playground Series S6E3](https://www.kaggle.com/competitions/playground-series-s6e3) leaderboard.

**Data.** The synthetic competition `train.csv` / `test.csv`, augmented with the well-known original *IBM Telco Customer Churn* dataset as extra signal for target-statistics features.

**Approach.** A single XGBoost configuration with heavy telco feature engineering, wrapped in a **seed ensemble**: we re-run the same cross-validated model under several different `random_state` values and **average** the out-of-fold (OOF) and test predictions. Averaging across seeds cancels the fold-assignment noise that any one split injects, lowering prediction variance and producing a more stable leaderboard score than any single run.

**Author:** Lorenzo Scaturchio

## 1. Objective & Introduction

The **goal** of this notebook is to turn a strong single XGBoost pipeline into a *low-variance* predictor by ensembling across random seeds, and to make the mechanism transparent rather than magical.

Why does seed averaging help? A gradient-boosted model on tabular data is not a deterministic function of the data alone. Several knobs are seeded:

- the **fold assignment** in `StratifiedKFold(shuffle=True, random_state=...)`,
- the **row subsampling** (`subsample`) drawn each boosting round,
- the **column subsampling** (`colsample_bytree`) at each tree.

Each seed therefore produces a slightly different model whose error has two parts: a *bias* component (systematic, shared across seeds) and a *variance* component (the random part that differs seed to seed). Averaging `N` approximately-independent predictions leaves the bias untouched but shrinks the variance term by roughly `1/N` when the runs are uncorrelated, and by less when they are correlated. Because all our seeds share the same data and hyper-parameters, the runs are *positively* correlated, so we expect a real but **diminishing** benefit as `N` grows - a trade-off we quantify later.

Concretely, this notebook will:

1. Load and explore the competition and original telco data.
2. Inspect the target balance and feature distributions with charts.
3. Define the **SEED list** that is the heart of the method.
4. Train the seed-averaged, nested target-encoded XGBoost ensemble.
5. Compare per-seed CV scores against the pooled ensemble score.
6. Interpret the variance reduction and write `submission.csv`.

## 2. Setup & Reproducibility

Reproducibility is not an afterthought here - it is the *subject* of the notebook. We fix a global `RANDOM_STATE` for any one-off operations (EDA sampling, plot jitter) and define an explicit `SEEDS` list that drives the ensemble. Every model and every data split downstream is seeded from one of these values, so the entire run is deterministic and re-runnable.

In [ ]:
import json
import warnings
from itertools import combinations
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

# --- Reproducibility controls -------------------------------------------
RANDOM_STATE = 42          # global seed for one-off EDA / plotting ops
np.random.seed(RANDOM_STATE)

# The SEED list is the core of the method: each value yields one full
# cross-validated XGBoost run, and we average across them.
SEEDS = (11, 42, 99)
print(f'Global RANDOM_STATE = {RANDOM_STATE}')
print(f'Ensemble SEEDS      = {SEEDS}  (N = {len(SEEDS)} runs)')
print('Environment ready.')

## 3. Data Overview & Loading

We resolve the input paths robustly because the same notebook runs both as a Kaggle kernel (data mounted under `/kaggle/input`) and in attached-dataset variants. Three frames are loaded:

- **`train`** - labelled competition rows (the churn target).
- **`test`** - unlabelled competition rows we must score.
- **`orig`** - the original IBM Telco dataset, used only to build *target-statistics* features (group churn rates, distribution ranks). It is never used as training labels, which keeps the validation honest.

In [ ]:
def find_input_file(filename: str) -> Path | None:
    candidates = [
        Path('/kaggle/input/playground-series-s6e3') / filename,
        Path('/kaggle/input/competitions/playground-series-s6e3') / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.rglob(filename))
        if matches:
            return matches[0]
    return None

def find_original_telco_file() -> Path | None:
    candidates = [
        Path('/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
        Path('/kaggle/input/wa-fnusec-telcocustomerchurn/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
        Path('/kaggle/input/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.rglob('WA_Fn-UseC_-Telco-Customer-Churn.csv'))
        if matches:
            return matches[0]
    return None

train_path = find_input_file('train.csv')
test_path = find_input_file('test.csv')
orig_path = find_original_telco_file()

if train_path is None or test_path is None or orig_path is None:
    raise FileNotFoundError('Required competition or original telco files were not found under /kaggle/input.')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
orig = pd.read_csv(orig_path)

print(f'train: {train.shape}')
print(f'test : {test.shape}')
print(f'orig : {orig.shape}')
print(f'competition train path: {train_path}')
print(f'original telco path  : {orig_path}')

### 3.1 Schema preview

Below we look at the column dtypes and a few sample rows so the feature space is concrete before we engineer on top of it. Telco data mixes a handful of numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`) with many low-cardinality categorical service flags.

In [ ]:
schema = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'n_unique': train.nunique(),
    'n_missing': train.isna().sum(),
})
print('Competition train schema:')
display(schema)
train.head()

## 4. Exploratory Data Analysis (EDA)

### 4.1 Target balance

ROC AUC is robust to class imbalance, but knowing the churn base rate still matters: it sets the reference point for `scale_pos_weight`-style decisions and tells us how informative the original-data target features can be. We normalise the raw `Yes`/`No` (or `1`/`0`) target to integers first.

In [ ]:
TARGET = 'Churn'

def to_binary_target(series: pd.Series) -> pd.Series:
    mapped = series.astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0})
    return mapped.fillna(pd.to_numeric(series, errors='coerce')).astype(int)

train_target = to_binary_target(train[TARGET])
orig_target = to_binary_target(orig[TARGET])
balance = pd.DataFrame({
    'competition_train': train_target.value_counts(normalize=True).sort_index(),
    'original_telco': orig_target.value_counts(normalize=True).sort_index(),
})
print('Churn rate (competition):', round(float(train_target.mean()), 4))
print('Churn rate (original)  :', round(float(orig_target.mean()), 4))

fig, ax = plt.subplots(figsize=(6.5, 4))
balance.plot(kind='bar', ax=ax)
ax.set_title('Churn class balance: competition vs original telco')
ax.set_xlabel('Churn (0 = stays, 1 = churns)')
ax.set_ylabel('Proportion')
ax.set_xticklabels(['stays (0)', 'churns (1)'], rotation=0)
ax.legend(title='dataset')
plt.tight_layout()
plt.show()

**Observation.** Churn is the minority class in both datasets (roughly a quarter of customers). The competition and original distributions are close but not identical, which is *why* the original data is useful as an auxiliary signal rather than as drop-in extra training rows - its target rates transfer, its exact row distribution does not.

### 4.2 Numeric feature distributions by churn

The three core numeric drivers are contract `tenure` and the two charge columns. We coerce them to numeric (the raw `TotalCharges` has blank strings) and compare their distributions for churners vs non-churners.

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
eda = train.copy()
eda[TARGET] = train_target.values
for col in num_cols:
    eda[col] = pd.to_numeric(eda[col], errors='coerce')
    eda[col] = eda[col].fillna(eda[col].median())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    for churn_val, label in [(0, 'stays'), (1, 'churns')]:
        sns.kdeplot(
            data=eda[eda[TARGET] == churn_val], x=col,
            ax=ax, fill=True, alpha=0.35, label=label,
        )
    ax.set_title(f'{col} by churn')
    ax.legend()
plt.tight_layout()
plt.show()

**Finding.** Churn concentrates at **low tenure and high monthly charges** - new customers on expensive month-to-month plans leave first. `TotalCharges` is almost a proxy for tenure (it accumulates over time), which is why the feature pipeline derives ratios like `monthly_to_total_ratio` and a `charges_deviation` term to separate *price level* from *time on book*.

### 4.3 Categorical churn drivers

Among the categorical service flags, `Contract` and `InternetService` are the strongest churn signals in telco data. We chart per-category churn rates to confirm before trusting them as target-encoding sources.

In [ ]:
cat_drivers = ['Contract', 'InternetService', 'PaymentMethod']
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, col in zip(axes, cat_drivers):
    rates = eda.groupby(col)[TARGET].mean().sort_values()
    sns.barplot(x=rates.values, y=rates.index, ax=ax, color='#c0392b')
    ax.axvline(train_target.mean(), color='k', ls='--', lw=1,
               label=f'base rate {train_target.mean():.2f}')
    ax.set_title(f'Churn rate by {col}')
    ax.set_xlabel('P(churn)')
    ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

**Interpretation.** Month-to-month contracts, fiber-optic internet, and electronic-check payments churn far above the base rate, while two-year contracts churn far below it. These large category-to-category gaps are exactly what target encoding exploits - and exactly why the encoding must be done *inside* cross-validation (next section) to avoid leaking the validation target.

## 5. Method: Feature Pipeline + Seed-Ensemble Model

### 5.1 Feature engineering & model functions

The cell below embeds the exact, battle-tested feature and model functions from the repo's competition lab so this kernel reproduces the local benchmark without re-implementation drift. The pipeline builds:

- frequency, rank, log/sqrt/inverse and ratio transforms of the numeric columns,
- service-count aggregates and binary `is-yes` / `is-no` flags,
- pairwise and triple categorical **interactions** plus rare-category counts,
- **original-data target statistics** (group churn means, distribution ranks),
- and a **nested, out-of-fold target encoder** so leakage is impossible.

`_playground_advanced_xgboost_result` is the engine: it loops over the `seeds` argument, runs a full `StratifiedKFold` per seed, and accumulates OOF and test predictions. Note how `random_state=seed` is threaded through *every* stochastic object (outer CV, inner CV, the target encoder, and the XGBoost model itself).

In [ ]:
def _concat_feature_block(df: pd.DataFrame, updates: dict[str, Any]) -> pd.DataFrame:
    if not updates:
        return df
    block = pd.DataFrame(updates, index=df.index)
    return pd.concat([df, block], axis=1).copy()


def _playground_advanced_feature_frames(
    train: pd.DataFrame,
    test: pd.DataFrame,
    orig: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str], list[str]]:
    def pctrank_against(values: np.ndarray, reference: np.ndarray) -> np.ndarray:
        ref = np.sort(np.asarray(reference, dtype=np.float32))
        if ref.size == 0:
            return np.zeros(len(values), dtype=np.float32)
        return (np.searchsorted(ref, values, side="left") / ref.size).astype(np.float32)

    def zscore_against(values: np.ndarray, reference: np.ndarray) -> np.ndarray:
        ref = np.asarray(reference, dtype=np.float32)
        if ref.size == 0:
            return np.zeros(len(values), dtype=np.float32)
        sigma = float(ref.std())
        if sigma == 0.0 or np.isnan(sigma):
            return np.zeros(len(values), dtype=np.float32)
        return ((values - float(ref.mean())) / sigma).astype(np.float32)

    train = train.copy()
    test = test.copy()
    orig = orig.copy()
    if "customerID" in orig.columns:
        orig = orig.drop(columns=["customerID"])

    target = "Churn"
    train[target] = (
        train[target].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0}).fillna(train[target]).astype(int)
    )
    orig[target] = (
        orig[target].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0}).fillna(orig[target]).astype(int)
    )
    cat_cols = [
        "gender",
        "SeniorCitizen",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
    ]
    num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
    service_cols = [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]

    for df in (train, test, orig):
        for col in cat_cols:
            df[col] = df[col].astype(str).fillna("missing").str.strip()
        for col in num_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")
            df[col] = df[col].fillna(df[col].median())

    new_num_cols: list[str] = []
    freq_maps = {
        col: pd.concat([train[col], test[col], orig[col]], axis=0).value_counts(normalize=True)
        for col in num_cols
    }
    train = _concat_feature_block(
        train,
        {f"FREQ_{col}": train[col].map(freq_maps[col]).fillna(0).astype("float32") for col in num_cols},
    )
    test = _concat_feature_block(
        test,
        {f"FREQ_{col}": test[col].map(freq_maps[col]).fillna(0).astype("float32") for col in num_cols},
    )
    orig = _concat_feature_block(
        orig,
        {f"FREQ_{col}": orig[col].map(freq_maps[col]).fillna(0).astype("float32") for col in num_cols},
    )
    new_num_cols.extend([f"FREQ_{col}" for col in num_cols])

    all_num = pd.concat([train[num_cols], test[num_cols], orig[num_cols]], axis=0, ignore_index=True)
    rank_updates_train: dict[str, Any] = {}
    rank_updates_test: dict[str, Any] = {}
    rank_updates_orig: dict[str, Any] = {}
    for col in num_cols:
        ranks = all_num[col].rank(method="average", pct=True).astype("float32").to_numpy()
        rank_updates_train[f"RANK_{col}"] = ranks[: len(train)]
        rank_updates_test[f"RANK_{col}"] = ranks[len(train) : len(train) + len(test)]
        rank_updates_orig[f"RANK_{col}"] = ranks[len(train) + len(test) :]
    train = _concat_feature_block(train, rank_updates_train)
    test = _concat_feature_block(test, rank_updates_test)
    orig = _concat_feature_block(orig, rank_updates_orig)
    new_num_cols.extend([f"RANK_{col}" for col in num_cols])

    def _power_updates(df: pd.DataFrame) -> dict[str, Any]:
        updates: dict[str, Any] = {}
        for col in num_cols:
            values = df[col].astype("float32")
            updates[f"LOG1P_{col}"] = np.log1p(values.clip(lower=0)).astype("float32")
            updates[f"SQRT_{col}"] = np.sqrt(values.clip(lower=0)).astype("float32")
            updates[f"INV1P_{col}"] = (1.0 / (1.0 + values.clip(lower=0))).astype("float32")
        return updates

    train = _concat_feature_block(train, _power_updates(train))
    test = _concat_feature_block(test, _power_updates(test))
    orig = _concat_feature_block(orig, _power_updates(orig))
    new_num_cols.extend([f"LOG1P_{col}" for col in num_cols])
    new_num_cols.extend([f"SQRT_{col}" for col in num_cols])
    new_num_cols.extend([f"INV1P_{col}" for col in num_cols])

    def _core_numeric_updates(df: pd.DataFrame) -> dict[str, Any]:
        charges_deviation = (df["TotalCharges"] - df["tenure"] * df["MonthlyCharges"]).astype("float32")
        service_yes_count = (df[service_cols] == "Yes").sum(axis=1).astype("float32")
        return {
            "charges_deviation": charges_deviation,
            "abs_charges_dev": np.abs(charges_deviation).astype("float32"),
            "monthly_to_total_ratio": (df["MonthlyCharges"] / (df["TotalCharges"] + 1)).astype("float32"),
            "total_to_monthly_ratio": (df["TotalCharges"] / (df["MonthlyCharges"] + 1)).astype("float32"),
            "avg_monthly_charges": (df["TotalCharges"] / (df["tenure"] + 1)).astype("float32"),
            "tenure_x_monthly": (df["tenure"] * df["MonthlyCharges"]).astype("float32"),
            "tenure_x_total": (df["tenure"] * df["TotalCharges"]).astype("float32"),
            "service_yes_count": service_yes_count,
            "service_no_count": (df[service_cols] == "No").sum(axis=1).astype("float32"),
            "service_other_count": (
                df[service_cols].isin(["No phone service", "No internet service"]).sum(axis=1).astype("float32")
            ),
            "service_count": service_yes_count,
            "has_internet": (df["InternetService"] != "No").astype("float32"),
            "has_phone": (df["PhoneService"] == "Yes").astype("float32"),
        }

    train = _concat_feature_block(train, _core_numeric_updates(train))
    test = _concat_feature_block(test, _core_numeric_updates(test))
    orig = _concat_feature_block(orig, _core_numeric_updates(orig))
    new_num_cols.extend(
        [
            "charges_deviation",
            "abs_charges_dev",
            "monthly_to_total_ratio",
            "total_to_monthly_ratio",
            "avg_monthly_charges",
            "tenure_x_monthly",
            "tenure_x_total",
            "service_yes_count",
            "service_no_count",
            "service_other_count",
            "service_count",
            "has_internet",
            "has_phone",
        ]
    )

    new_cat_cols: list[str] = []
    tenure_bins = [0, 1, 3, 6, 12, 24, 36, 48, 60, 72, 10_000]
    monthly_bins = pd.qcut(
        pd.concat([train["MonthlyCharges"], test["MonthlyCharges"], orig["MonthlyCharges"]]),
        q=40,
        retbins=True,
        duplicates="drop",
    )[1]
    total_bins = pd.qcut(
        pd.concat([train["TotalCharges"], test["TotalCharges"], orig["TotalCharges"]]),
        q=60,
        retbins=True,
        duplicates="drop",
    )[1]
    train = _concat_feature_block(
        train,
        {
            "tenure_bin": pd.cut(train["tenure"], bins=tenure_bins, include_lowest=True).astype(str),
            "MonthlyCharges_bin": pd.cut(train["MonthlyCharges"], bins=monthly_bins, include_lowest=True).astype(str),
            "TotalCharges_bin": pd.cut(train["TotalCharges"], bins=total_bins, include_lowest=True).astype(str),
        },
    )
    test = _concat_feature_block(
        test,
        {
            "tenure_bin": pd.cut(test["tenure"], bins=tenure_bins, include_lowest=True).astype(str),
            "MonthlyCharges_bin": pd.cut(test["MonthlyCharges"], bins=monthly_bins, include_lowest=True).astype(str),
            "TotalCharges_bin": pd.cut(test["TotalCharges"], bins=total_bins, include_lowest=True).astype(str),
        },
    )
    orig = _concat_feature_block(
        orig,
        {
            "tenure_bin": pd.cut(orig["tenure"], bins=tenure_bins, include_lowest=True).astype(str),
            "MonthlyCharges_bin": pd.cut(orig["MonthlyCharges"], bins=monthly_bins, include_lowest=True).astype(str),
            "TotalCharges_bin": pd.cut(orig["TotalCharges"], bins=total_bins, include_lowest=True).astype(str),
        },
    )
    new_cat_cols.extend(["tenure_bin", "MonthlyCharges_bin", "TotalCharges_bin"])

    yn_cols = [
        "Partner",
        "Dependents",
        "PhoneService",
        "PaperlessBilling",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "MultipleLines",
    ]
    def _yn_updates(df: pd.DataFrame) -> dict[str, Any]:
        updates: dict[str, Any] = {}
        for col in yn_cols:
            values = df[col].astype(str)
            updates[f"ISYES_{col}"] = (values == "Yes").astype("float32")
            updates[f"ISNO_{col}"] = (values == "No").astype("float32")
            updates[f"ISOTHER_{col}"] = (~values.isin(["Yes", "No"])).astype("float32")
        return updates

    train = _concat_feature_block(train, _yn_updates(train))
    test = _concat_feature_block(test, _yn_updates(test))
    orig = _concat_feature_block(orig, _yn_updates(orig))
    new_num_cols.extend([f"ISYES_{col}" for col in yn_cols])
    new_num_cols.extend([f"ISNO_{col}" for col in yn_cols])
    new_num_cols.extend([f"ISOTHER_{col}" for col in yn_cols])

    cat_feature_updates = {"train": {}, "test": {}, "orig": {}}
    for left, right in (
        ("Contract", "InternetService"),
        ("PaymentMethod", "Contract"),
        ("InternetService", "OnlineSecurity"),
        ("PaymentMethod", "PaperlessBilling"),
        ("Contract", "PaperlessBilling"),
        ("InternetService", "TechSupport"),
    ):
        name = f"{left}__{right}"
        cat_feature_updates["train"][name] = train[left].astype(str) + "|" + train[right].astype(str)
        cat_feature_updates["test"][name] = test[left].astype(str) + "|" + test[right].astype(str)
        cat_feature_updates["orig"][name] = orig[left].astype(str) + "|" + orig[right].astype(str)
        new_cat_cols.append(name)

    for left, middle, right in (("Contract", "InternetService", "PaymentMethod"),):
        name = f"{left}__{middle}__{right}"
        cat_feature_updates["train"][name] = (
            train[left].astype(str) + "|" + train[middle].astype(str) + "|" + train[right].astype(str)
        )
        cat_feature_updates["test"][name] = (
            test[left].astype(str) + "|" + test[middle].astype(str) + "|" + test[right].astype(str)
        )
        cat_feature_updates["orig"][name] = (
            orig[left].astype(str) + "|" + orig[middle].astype(str) + "|" + orig[right].astype(str)
        )
        new_cat_cols.append(name)

    ngram_top_cols = [
        "Contract",
        "InternetService",
        "PaymentMethod",
        "OnlineSecurity",
        "TechSupport",
        "PaperlessBilling",
    ]
    for left, right in combinations(ngram_top_cols, 2):
        name = f"BG_{left}_{right}"
        cat_feature_updates["train"][name] = train[left].astype(str) + "_" + train[right].astype(str)
        cat_feature_updates["test"][name] = test[left].astype(str) + "_" + test[right].astype(str)
        cat_feature_updates["orig"][name] = orig[left].astype(str) + "_" + orig[right].astype(str)
        new_cat_cols.append(name)

    for left, middle, right in combinations(ngram_top_cols[:4], 3):
        name = f"TG_{left}_{middle}_{right}"
        cat_feature_updates["train"][name] = (
            train[left].astype(str) + "_" + train[middle].astype(str) + "_" + train[right].astype(str)
        )
        cat_feature_updates["test"][name] = (
            test[left].astype(str) + "_" + test[middle].astype(str) + "_" + test[right].astype(str)
        )
        cat_feature_updates["orig"][name] = (
            orig[left].astype(str) + "_" + orig[middle].astype(str) + "_" + orig[right].astype(str)
        )
        new_cat_cols.append(name)
    train = _concat_feature_block(train, cat_feature_updates["train"])
    test = _concat_feature_block(test, cat_feature_updates["test"])
    orig = _concat_feature_block(orig, cat_feature_updates["orig"])

    counted_cat_cols = cat_cols + new_cat_cols
    all_cat_frame = pd.concat([train[counted_cat_cols], test[counted_cat_cols], orig[counted_cat_cols]], ignore_index=True)
    count_updates_train: dict[str, Any] = {}
    count_updates_test: dict[str, Any] = {}
    count_updates_orig: dict[str, Any] = {}
    for col in counted_cat_cols:
        counts = all_cat_frame[col].value_counts(dropna=False)
        train_counts = train[col].map(counts).fillna(0).astype("float32")
        test_counts = test[col].map(counts).fillna(0).astype("float32")
        orig_counts = orig[col].map(counts).fillna(0).astype("float32")
        count_updates_train[f"CAT_CNT_{col}"] = train_counts
        count_updates_test[f"CAT_CNT_{col}"] = test_counts
        count_updates_orig[f"CAT_CNT_{col}"] = orig_counts
        count_updates_train[f"CAT_RARE_{col}"] = (train_counts <= 50).astype("float32")
        count_updates_test[f"CAT_RARE_{col}"] = (test_counts <= 50).astype("float32")
        count_updates_orig[f"CAT_RARE_{col}"] = (orig_counts <= 50).astype("float32")
        new_num_cols.extend([f"CAT_CNT_{col}", f"CAT_RARE_{col}"])
    train = _concat_feature_block(train, count_updates_train)
    test = _concat_feature_block(test, count_updates_test)
    orig = _concat_feature_block(orig, count_updates_orig)

    orig_global = float(orig[target].mean())
    orig_proba_updates_train: dict[str, Any] = {}
    orig_proba_updates_test: dict[str, Any] = {}
    orig_proba_updates_orig: dict[str, Any] = {}
    for col in cat_cols + num_cols + new_cat_cols:
        lookup = orig.groupby(col, observed=False)[target].mean()
        name = f"ORIG_proba_{col}"
        orig_proba_updates_train[name] = train[col].map(lookup).fillna(orig_global).astype("float32")
        orig_proba_updates_test[name] = test[col].map(lookup).fillna(orig_global).astype("float32")
        orig_proba_updates_orig[name] = orig[col].map(lookup).fillna(orig_global).astype("float32")
        new_num_cols.append(name)
    train = _concat_feature_block(train, orig_proba_updates_train)
    test = _concat_feature_block(test, orig_proba_updates_test)
    orig = _concat_feature_block(orig, orig_proba_updates_orig)

    orig_churner_tc = orig.loc[orig[target] == 1, "TotalCharges"].to_numpy(dtype=np.float32)
    orig_nonchurner_tc = orig.loc[orig[target] == 0, "TotalCharges"].to_numpy(dtype=np.float32)
    orig_tc = orig["TotalCharges"].to_numpy(dtype=np.float32)
    orig_is_mc_mean = orig.groupby("InternetService", observed=False)["MonthlyCharges"].mean()
    distribution_cols = [
        "pctrank_nonchurner_TC",
        "pctrank_churner_TC",
        "pctrank_orig_TC",
        "zscore_churn_gap_TC",
        "zscore_nonchurner_TC",
        "pctrank_churn_gap_TC",
        "resid_IS_MC",
        "cond_pctrank_IS_TC",
        "cond_pctrank_C_TC",
    ]
    def _distribution_updates(df: pd.DataFrame) -> dict[str, Any]:
        tc = df["TotalCharges"].to_numpy(dtype=np.float32)
        updates: dict[str, Any] = {
            "pctrank_nonchurner_TC": pctrank_against(tc, orig_nonchurner_tc),
            "pctrank_churner_TC": pctrank_against(tc, orig_churner_tc),
            "pctrank_orig_TC": pctrank_against(tc, orig_tc),
            "zscore_churn_gap_TC": (
                np.abs(zscore_against(tc, orig_churner_tc)) - np.abs(zscore_against(tc, orig_nonchurner_tc))
            ).astype(np.float32),
            "zscore_nonchurner_TC": zscore_against(tc, orig_nonchurner_tc),
            "pctrank_churn_gap_TC": (
                pctrank_against(tc, orig_churner_tc) - pctrank_against(tc, orig_nonchurner_tc)
            ).astype(np.float32),
            "resid_IS_MC": (
                df["MonthlyCharges"] - df["InternetService"].map(orig_is_mc_mean).fillna(0).to_numpy(dtype=np.float32)
            ).astype(np.float32),
        }
        cond_is_vals = np.zeros(len(df), dtype=np.float32)
        for cat_val in orig["InternetService"].dropna().astype(str).unique():
            mask = df["InternetService"].astype(str) == cat_val
            if not mask.any():
                continue
            ref = orig.loc[orig["InternetService"].astype(str) == cat_val, "TotalCharges"].to_numpy(dtype=np.float32)
            cond_is_vals[mask.to_numpy()] = pctrank_against(
                df.loc[mask, "TotalCharges"].to_numpy(dtype=np.float32),
                ref,
            )
        updates["cond_pctrank_IS_TC"] = cond_is_vals

        cond_contract_vals = np.zeros(len(df), dtype=np.float32)
        for cat_val in orig["Contract"].dropna().astype(str).unique():
            mask = df["Contract"].astype(str) == cat_val
            if not mask.any():
                continue
            ref = orig.loc[orig["Contract"].astype(str) == cat_val, "TotalCharges"].to_numpy(dtype=np.float32)
            cond_contract_vals[mask.to_numpy()] = pctrank_against(
                df.loc[mask, "TotalCharges"].to_numpy(dtype=np.float32),
                ref,
            )
        updates["cond_pctrank_C_TC"] = cond_contract_vals
        return updates

    train = _concat_feature_block(train, _distribution_updates(train))
    test = _concat_feature_block(test, _distribution_updates(test))
    new_num_cols.extend(distribution_cols)

    num_as_cat: list[str] = []
    num_as_cat_updates_train: dict[str, Any] = {}
    num_as_cat_updates_test: dict[str, Any] = {}
    num_as_cat_updates_orig: dict[str, Any] = {}
    for col in num_cols:
        cat_name = f"CAT_{col}"
        num_as_cat.append(cat_name)
        num_as_cat_updates_train[cat_name] = train[col].astype(str)
        num_as_cat_updates_test[cat_name] = test[col].astype(str)
        num_as_cat_updates_orig[cat_name] = orig[col].astype(str)
    train = _concat_feature_block(train, num_as_cat_updates_train)
    test = _concat_feature_block(test, num_as_cat_updates_test)
    orig = _concat_feature_block(orig, num_as_cat_updates_orig)

    for df in (train, test, orig):
        for col in cat_cols + new_cat_cols + num_as_cat:
            df[col] = df[col].astype("category")

    feature_cols = num_cols + cat_cols + new_num_cols + new_cat_cols + num_as_cat
    te_cols = num_as_cat + cat_cols + new_cat_cols
    drop_raw_cols = num_as_cat + cat_cols + new_cat_cols
    return train, test, feature_cols, te_cols, drop_raw_cols


def _playground_advanced_xgboost_result(
    train: pd.DataFrame,
    test: pd.DataFrame,
    orig: pd.DataFrame,
    folds: int,
    seeds: tuple[int, ...] = (11, 42, 99),
) -> tuple[float, np.ndarray, np.ndarray]:
    target = "Churn"
    train_frame, test_frame, feature_cols, te_cols, drop_raw_cols = _playground_advanced_feature_frames(
        train,
        test,
        orig,
    )
    n_splits = min(max(3, folds), 5)
    inner_splits = min(3, n_splits)
    stats = ["std", "min", "max"]
    oof_sum = np.zeros(len(train_frame), dtype=float)
    oof_count = np.zeros(len(train_frame), dtype=float)
    test_pred = np.zeros(len(test_frame), dtype=float)
    total_models = 0

    try:
        import xgboost as xgb
    except ImportError as exc:
        raise RuntimeError("xgboost is not installed") from exc

    params = {
        "n_estimators": 6000,
        "learning_rate": 0.02,
        "max_depth": 5,
        "subsample": 0.81,
        "colsample_bytree": 0.55,
        "min_child_weight": 6,
        "reg_alpha": 1.25,
        "reg_lambda": 1.3,
        "gamma": 0.35,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "enable_categorical": True,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "early_stopping_rounds": 200,
    }

    for seed in seeds:
        outer_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for train_idx, valid_idx in outer_cv.split(train_frame, train_frame[target]):
            x_train = train_frame.iloc[train_idx][feature_cols + [target]].reset_index(drop=True).copy()
            y_train = train_frame.iloc[train_idx][target].to_numpy()
            y_valid = train_frame.iloc[valid_idx][target].to_numpy()
            x_valid = train_frame.iloc[valid_idx][feature_cols].reset_index(drop=True).copy()
            x_test = test_frame[feature_cols].reset_index(drop=True).copy()
            inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed)

            te_stat_cols = [f"TE1_{col}_{stat}" for col in te_cols for stat in stats]
            x_train = _concat_feature_block(x_train, {name: np.nan for name in te_stat_cols})

            for inner_train_idx, inner_valid_idx in inner_cv.split(x_train, y_train):
                x_inner_train = x_train.loc[inner_train_idx, feature_cols + [target]].copy()
                x_inner_valid = x_train.loc[inner_valid_idx, feature_cols].copy()
                for col in te_cols:
                    grouped = x_inner_train.groupby(col, observed=False)[target].agg(stats)
                    grouped.columns = [f"TE1_{col}_{stat}" for stat in stats]
                    x_inner_valid = x_inner_valid.merge(grouped, on=col, how="left")
                    for name in grouped.columns:
                        x_train.loc[inner_valid_idx, name] = x_inner_valid[name].to_numpy(dtype="float32")

            for col in te_cols:
                grouped = x_train.groupby(col, observed=False)[target].agg(stats)
                grouped.columns = [f"TE1_{col}_{stat}" for stat in stats]
                x_valid = x_valid.merge(grouped.astype("float32"), on=col, how="left")
                x_test = x_test.merge(grouped.astype("float32"), on=col, how="left")
                for name in grouped.columns:
                    x_train[name] = x_train[name].fillna(0).astype("float32")
                    x_valid[name] = x_valid[name].fillna(0).astype("float32")
                    x_test[name] = x_test[name].fillna(0).astype("float32")

            if te_cols:
                mean_encoder = TargetEncoder(
                    cv=inner_splits,
                    shuffle=True,
                    smooth="auto",
                    target_type="binary",
                    random_state=seed,
                )
                mean_cols = [f"TE_{col}" for col in te_cols]
                x_train = pd.concat(
                    [
                        x_train,
                        pd.DataFrame(
                            mean_encoder.fit_transform(x_train[te_cols], y_train),
                            columns=mean_cols,
                            index=x_train.index,
                        ),
                    ],
                    axis=1,
                ).copy()
                x_valid = pd.concat(
                    [
                        x_valid,
                        pd.DataFrame(
                            mean_encoder.transform(x_valid[te_cols]),
                            columns=mean_cols,
                            index=x_valid.index,
                        ),
                    ],
                    axis=1,
                ).copy()
                x_test = pd.concat(
                    [
                        x_test,
                        pd.DataFrame(
                            mean_encoder.transform(x_test[te_cols]),
                            columns=mean_cols,
                            index=x_test.index,
                        ),
                    ],
                    axis=1,
                ).copy()

            for df in (x_train, x_valid, x_test):
                for col in te_cols:
                    df[col] = df[col].astype(str).astype("category")
                df.drop(columns=drop_raw_cols, inplace=True)
            x_train.drop(columns=[target], inplace=True)

            model = xgb.XGBClassifier(**params, random_state=seed)
            model.fit(
                x_train,
                y_train,
                eval_set=[(x_valid, y_valid)],
                verbose=False,
            )
            valid_pred = model.predict_proba(x_valid)[:, 1]
            oof_sum[valid_idx] += valid_pred
            oof_count[valid_idx] += 1.0
            test_pred += model.predict_proba(x_test)[:, 1]
            total_models += 1

    if total_models == 0 or np.any(oof_count == 0):
        raise RuntimeError("XGBoost did not produce a complete OOF prediction.")

    oof = oof_sum / oof_count
    test_pred = test_pred / total_models
    return float(roc_auc_score(train_frame[target].to_numpy(), oof)), oof, test_pred


### 5.2 Single-model baseline vs the ensemble - the bias-variance picture

Before averaging anything, it helps to state the bias-variance trade-off in this concrete setting. A single seeded run gives an unbiased-ish but *noisy* estimate of the true churn probability for each customer; the noise comes from which rows happened to land in which fold and which columns each tree happened to see. If we write the per-seed prediction as `true_signal + seed_noise`, then averaging `N` seeds keeps `true_signal` and averages the noise toward zero.

The variance reduction is `Var_ensemble = rho*Var_single + (1-rho)/N*Var_single`, where `rho` is the average correlation between seed predictions. Because our seeds share data and hyper-parameters, `rho` is high (typically 0.9+), so the floor `rho*Var_single` dominates - **therefore** the first few seeds buy most of the stability and additional seeds show clear diminishing returns. We will see this directly in the per-seed score spread.

## 6. Seed-Ensemble Training

We now run the full ensemble across `SEEDS`. The function returns the pooled OOF AUC, the OOF prediction vector, and the averaged test prediction. This is the heavy cell - on Kaggle it trains `len(SEEDS) x folds` boosted models.

In [ ]:
score, oof, pred = _playground_advanced_xgboost_result(
    train, test, orig, folds=5, seeds=SEEDS,
)
summary = {
    'n_seeds': len(SEEDS),
    'ensemble_oof_auc': round(float(score), 5),
    'prediction_rows': int(len(pred)),
    'prediction_min': round(float(pred.min()), 5),
    'prediction_max': round(float(pred.max()), 5),
    'prediction_mean': round(float(pred.mean()), 5),
}
print(json.dumps(summary, indent=2))

## 7. Results & Evaluation

### 7.1 Per-seed CV scores vs the pooled ensemble

To make the variance-reduction claim measurable, we score **each seed on its own** by re-running the engine with a single-element seed list, then compare the spread of those per-seed AUCs against the pooled ensemble AUC. The pooled score should sit at or above the per-seed mean with materially lower run-to-run variance - that gap is the entire value proposition of the method.

In [ ]:
per_seed_scores = {}
for s in SEEDS:
    s_auc, _, _ = _playground_advanced_xgboost_result(
        train, test, orig, folds=5, seeds=(s,),
    )
    per_seed_scores[s] = float(s_auc)
    print(f'seed {s:>3}: single-seed OOF AUC = {s_auc:.5f}')

seed_auc = np.array(list(per_seed_scores.values()))
print('-' * 44)
print(f'per-seed mean AUC : {seed_auc.mean():.5f}')
print(f'per-seed std  AUC : {seed_auc.std(ddof=0):.5f}')
print(f'ensemble    AUC : {score:.5f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
x = list(range(len(SEEDS)))
ax.scatter(x, seed_auc, s=90, color='#2980b9', zorder=3, label='single-seed AUC')
ax.axhline(seed_auc.mean(), color='#7f8c8d', ls=':', lw=1.5,
           label=f'per-seed mean {seed_auc.mean():.4f}')
ax.axhline(score, color='#27ae60', ls='--', lw=2,
           label=f'ensemble {score:.4f}')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in SEEDS])
ax.set_xlabel('random seed')
ax.set_ylabel('OOF ROC AUC')
ax.set_title('Per-seed CV scores vs pooled seed-ensemble')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

**Observation.** The single-seed points scatter around their mean, while the ensemble line sits at the top of (or above) that cloud. The ensemble does not merely *match* the average seed - by cancelling fold-assignment noise it typically edges past the best single seed, **because** the averaged OOF vector is a smoother, lower-variance estimate of each customer's churn probability.

### 7.2 ROC curve of the pooled OOF predictions

The OOF predictions cover every training row exactly once (each row scored by the folds in which it was held out), so the OOF ROC curve is an honest, leakage-free estimate of leaderboard behaviour.

In [ ]:
fpr, tpr, _ = roc_curve(train_target.to_numpy(), oof)
fig, ax = plt.subplots(figsize=(5.8, 5.4))
ax.plot(fpr, tpr, lw=2.2, color='#8e44ad', label=f'ensemble OOF (AUC={score:.4f})')
ax.plot([0, 1], [0, 1], ls='--', color='gray', lw=1, label='random')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curve - seed-ensemble out-of-fold predictions')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 7.3 OOF probability distribution by true class

A final sanity check: a well-calibrated ranking model should push churners toward high predicted probabilities and non-churners toward low ones, with visible separation between the two distributions.

In [ ]:
oof_df = pd.DataFrame({'p_churn': oof, 'actual': train_target.to_numpy()})
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for churn_val, label, color in [(0, 'stays', '#3498db'), (1, 'churns', '#e74c3c')]:
    sns.kdeplot(
        data=oof_df[oof_df['actual'] == churn_val], x='p_churn',
        ax=ax, fill=True, alpha=0.4, label=label, color=color,
    )
ax.set_title('Ensemble OOF predicted churn probability by true class')
ax.set_xlabel('predicted P(churn)')
ax.legend()
plt.tight_layout()
plt.show()

**Finding.** The two class distributions are clearly displaced - non-churners pile up at low probabilities and churners shift right - confirming the model ranks well, which is what AUC rewards. The overlap in the middle is the irreducible ambiguity that no amount of seed averaging can remove, since averaging shrinks *variance*, not *bias*.

## 8. Submission

We write the averaged test predictions to `submission.csv` in the required `id, Churn` format. Because `pred` is the mean over all seeds and folds, this file is the low-variance ensemble output rather than any single run.

In [ ]:
submission = pd.DataFrame({'id': test['id'], 'Churn': pred})
submission.to_csv('submission.csv', index=False)
print('submission.csv written to the working directory.')
submission.head()

## 9. Insights, Limitations & Caveats

**What the seed ensemble bought us.**

- *Variance reduction.* The pooled ensemble AUC is at least as high as the per-seed mean and, more importantly, would barely move if we changed the global seed - that stability is the headline insight. A single-seed pipeline can win or lose a leaderboard rank purely on a lucky fold split; the ensemble removes most of that luck.
- *Diminishing returns.* Because the per-seed predictions are highly correlated (`rho` near 1), the variance floor is `rho * Var_single`. Going from one to three seeds captures most of the achievable reduction; going from three to ten would shave only a little more at three times the compute - a clear **trade-off** between stability and runtime.

**Limitations and caveats.**

- *Bias is untouched.* Averaging cannot fix a mis-specified model or a leaky feature; it only smooths variance. If the single model is biased, the ensemble is biased by the same amount - a key **limitation**.
- *Correlated seeds.* Our seeds vary only the random state, not the architecture. A more powerful (but heavier) ensemble would also vary hyper-parameters or model families to *lower* `rho` and break through the variance floor.
- *Compute cost.* Each seed is a full nested-CV training run; the **caveat** is that wall-clock time scales linearly with the number of seeds, so the seed count should be chosen against the diminishing-returns curve, not maximised blindly.
- *Hypothesis for further gains.* Replacing seed averaging with a small, diverse model zoo (XGBoost + LightGBM + CatBoost) would likely beat pure seed averaging, **because** lower inter-model correlation pushes the variance floor down further.

## 10. Conclusion & Next Steps

**Summary.** We framed telco churn as an ROC-AUC ranking problem, engineered a rich telco feature set with leakage-safe nested target encoding, and wrapped a single strong XGBoost configuration in a **seed ensemble**. By averaging out-of-fold and test predictions across the explicit `SEEDS` list, we traded a small, fixed amount of extra compute for a measurably more stable leaderboard score - the per-seed spread chart and the pooled-vs-mean comparison make the variance reduction concrete.

**Key takeaways.**

- Seed averaging shrinks *variance*, not *bias*; the first few seeds deliver most of the benefit.
- Reproducibility is achievable end-to-end by threading `random_state` through every stochastic component.
- Honest OOF evaluation (ROC, score spread, probability separation) is what lets us trust the gain instead of guessing at it.

**Next steps / future work.**

1. **Diversify the ensemble** - add LightGBM and CatBoost runs so inter-model correlation drops and the variance floor falls further; we recommend this as the highest-leverage improvement.
2. **Tune the seed count** against a measured diminishing-returns curve rather than a fixed guess, to spend compute where it still helps.
3. **Probability calibration** (isotonic / Platt) on the pooled OOF output if a downstream decision threshold (not just ranking) is ever required.
4. **Stacking** - feed the per-seed OOF columns into a lightweight meta-learner to improve on plain averaging.

These are concrete, prioritised directions to push the score beyond what pure seed averaging can reach.